##### CODICE PER LA RICERCA DEI TEMPI DI PERCORRENZA A PARTIRE DAL CENTROIDE VERSO I PRONTO SOCCORSO, AEROPORTI, PORTI E STAZIONI FERROVIARIE

CARICAMENTO PACCHETTI

In [ ]:
import pandas as pd
import geopandas as gpd
import orjson
import asyncio
import aiohttp
import random
import nest_asyncio
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from math import radians, cos, sin, asin, sqrt

SET WORKING DIRECTORY

In [ ]:
import os
from pathlib import Path

# Get the current notebook's directory
notebook_dir = Path().absolute()

# Get the parent directory of the current notebook's folder
parent_dir = notebook_dir.parent

# Change the working directory to the parent folder
os.chdir(parent_dir)

# Print to verify the change
print("New Working Directory:", os.getcwd())

**OSPEDALI**

PULIZIA DEI DATASET 1

In [ ]:
import pandas as pd

# Percorso del file CSV
csv_path = "DATASET_DIST\hosp_lat_long.csv"

# Prova a leggere il file con diversi delimitatori
try:
    df = pd.read_csv(csv_path, delimiter=",", quotechar='"', skipinitialspace=True, encoding="utf-8", keep_default_na=False)
except Exception as e:
    print(f"Errore nel caricamento del file: {e}")
    exit()

# Stampiamo i nomi delle colonne per verificare eventuali problemi
print("Nomi colonne dopo la lettura:", df.columns.tolist())

# Normalizziamo i nomi delle colonne per evitare errori
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace(".", "_")

# Stampiamo di nuovo i nomi delle colonne dopo la pulizia
print("Nomi colonne dopo la pulizia:", df.columns.tolist())

# Sostituisci con i nomi corretti che trovi dopo la stampa
colonne_da_selezionare = ['Codice_struttura', 'Subcodice', 'latitude', 'longitude', 'Sigla_provincia']

# Controlliamo se le colonne esistono
colonne_presenti = [col for col in colonne_da_selezionare if col in df.columns]
colonne_mancanti = [col for col in colonne_da_selezionare if col not in df.columns]

if colonne_mancanti:
    print(f"⚠️ Attenzione! Queste colonne non sono state trovate nel CSV: {colonne_mancanti}")
    print("Assicurati che i nomi delle colonne siano scritti correttamente.")
    exit()

# Selezioniamo solo le colonne necessarie
df_selected = df[colonne_presenti]

# Assicuriamoci che "NA" (Napoli) non venga convertito in NaN
df_selected['Sigla_provincia'] = df_selected['Sigla_provincia'].fillna("NA")

# Visualizza i primi risultati
print(df_selected.head())

# Salva il risultato in un nuovo file CSV, se necessario
output_path = "DATASET_DIST\pronto_soccorso_lat_long.csv"
df_selected.to_csv(output_path, index=False)

print(f"Dati salvati con successo in: {output_path}")


PULIZIA DEI DATASET 2

In [ ]:
# Carica dataset e rinomina le colonne per uniformità
centroids_df = pd.read_csv("DATASET_DIST/centroidi_parte_2.csv", dtype=str)
centroids_df.rename(columns=lambda x: x.strip(), inplace=True)  # Rimuove spazi indesiderati
centroids_df.rename(columns={"provincia": "Sigla_provincia"}, inplace=True)

hospitals_df = pd.read_csv("DATASET_DIST/pronto_soccorso_lat_long.csv", dtype=str, delimiter=",")
hospitals_df.rename(columns=lambda x: x.strip(), inplace=True)  # Rimuove spazi indesiderati

province_df = pd.read_csv("DATASET_DIST/prov_conf.csv", dtype=str)
province_df.rename(columns=lambda x: x.strip(), inplace=True)
province_df.rename(columns={"Provincia": "Sigla_provincia"}, inplace=True)

# Stampa i nomi delle colonne per verifica
print("Centroids Colonne:", centroids_df.columns.tolist())
print("Hospitals Colonne:", hospitals_df.columns.tolist())
print("Province Colonne:", province_df.columns.tolist())

# Verifica che "Sigla_provincia" sia presente
assert "Sigla_provincia" in centroids_df.columns, "Errore: 'Sigla_provincia' mancante in centroidi"
assert "Sigla_provincia" in province_df.columns, "Errore: 'Sigla_provincia' mancante in province"

FUNZIONE PER LA RICERCA NELLA PROVINCIA E CONFINANTI

In [ ]:
def get_province_confinanti(prov, province_df, province_neighbors_dict=None):
    """
    Restituisce una lista con la provincia specificata e le province confinanti.
    
    Ottimizzato per l'accesso rapido tramite un dizionario che mappa province a province confinanti.

    Parameters:
        province_df (pd.DataFrame): DataFrame con la colonna 'Sigla_provincia' e le province confinanti come colonne successive.
        prov (str): Sigla della provincia (es. 'BG').
        province_neighbors_dict (dict, opzionale): Dizionario che mappa province alle province confinanti.

    Returns:
        list: Lista della provincia e delle sue confinanti.
    """
    if province_neighbors_dict is None:
        # Costruisci il dizionario una sola volta se non è stato passato
        province_neighbors_dict = {}
        for _, row in province_df.iterrows():
            prov_sigla = row['Sigla_provincia']
            confinanti = row.iloc[1:].dropna().values.tolist()
            province_neighbors_dict[prov_sigla] = [prov_sigla] + confinanti
    
    # Recupera la lista delle province confinanti in O(1)
    if prov not in province_neighbors_dict:
        raise ValueError(f"La provincia '{prov}' non è presente nel DataFrame.")
    
    return province_neighbors_dict[prov]


RICERCA DEI PRONTO SOCCORSO PIU VICINI AL CENTROIDE

In [ ]:
# Evita errori di event loop in Jupyter
nest_asyncio.apply()


# URL OSRM
OSRM_URL = "http://localhost:5000/route/v1/driving"

# Configurazione della concorrenza
SEMAPHORE = asyncio.Semaphore(50)  # Limitazione delle richieste simultanee
cache = {}
TIMEOUT = 600  # Timeout globale di 10 minuti

async def get_travel_time(session, coords, max_retries=3):
    """Effettua una richiesta OSRM per calcolare il tempo di viaggio con gestione dei timeout"""
    key = tuple(coords)
    if key in cache:
        return cache[key]

    url = f"{OSRM_URL}/{coords[1]},{coords[0]};{coords[3]},{coords[2]}?overview=false"

    for attempt in range(max_retries):
        async with SEMAPHORE:
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=TIMEOUT)) as response:
                    if response.status == 200:
                        data = orjson.loads(await response.read())
                        if 'routes' in data and data['routes']:
                            duration = data['routes'][0]['duration'] / 60  # Converti secondi in minuti
                            cache[key] = duration
                            return duration
            except (asyncio.TimeoutError, aiohttp.ClientError) as e:
                print(f"⚠️ Timeout o errore di rete ({e}). Ritento {attempt+1}/{max_retries}...") 
                await asyncio.sleep(2 ** attempt)  # Backoff esponenziale

    return None

async def process_centroid(session, centroid, hospitals, province_df, province_neighbors_dict):
    """Trova l'ospedale più vicino e il secondo più vicino per un centroide"""
    lat1, lon1, prov = centroid['latitudine'], centroid['longitudine'], centroid['Sigla_provincia']

    # Cerca ospedali nella provincia e province confinanti usando il dizionario
    province_da_cercare = get_province_confinanti(prov, province_df, province_neighbors_dict)
    same_prov_df = hospitals[hospitals['Sigla_provincia'].isin(province_da_cercare)]

    if same_prov_df.empty:
        return {
            "grid_latitude": lat1,
            "grid_longitude": lon1,
            "grid_SIGLA_PROVINCIA": prov,
            "ps_code": None,
            "ps_subcode": None,
            "ps_latitude": None,
            "ps_longitude": None,
            "travel_time_ps": None,
            "2ps_code": None,
            "2ps_subcode": None,
            "2ps_latitude": None,
            "2ps_longitude": None,
            "travel_time_2ps": None
        }

    coords_list = [(lat1, lon1, row['latitude'], row['longitude']) for _, row in same_prov_df.iterrows()]
    tasks = [get_travel_time(session, coords) for coords in coords_list]

    travel_times = await asyncio.gather(*tasks)
    indexed_times = [(i, t) for i, t in enumerate(travel_times) if t is not None]
    if len(indexed_times) == 0:
        return {
            "grid_latitude": lat1,
            "grid_longitude": lon1,
            "grid_SIGLA_PROVINCIA": prov,
            "ps_code": None,
            "ps_subcode": None,
            "ps_latitude": None,
            "ps_longitude": None,
            "travel_time_ps": None,
            "2ps_code": None,
            "2ps_subcode": None,
            "2ps_latitude": None,
            "2ps_longitude": None,
            "travel_time_2ps": None
        }

    # Ordina in base al tempo di percorrenza
    sorted_times = sorted(indexed_times, key=lambda x: x[1])
    first_idx, first_time = sorted_times[0]
    second_idx, second_time = sorted_times[1] if len(sorted_times) > 1 else (None, None)

    first_hospital = same_prov_df.iloc[first_idx]
    second_hospital = same_prov_df.iloc[second_idx] if second_idx is not None else None

    return {
        "grid_latitude": lat1,
        "grid_longitude": lon1,
        "grid_SIGLA_PROVINCIA": prov,
        "ps_code": first_hospital['Codice_struttura'],
        "ps_subcode": first_hospital['Subcodice'],
        "ps_latitude": first_hospital['latitude'],
        "ps_longitude": first_hospital['longitude'],
        "travel_time_ps": first_time,
        "2ps_code": second_hospital['Codice_struttura'] if second_hospital is not None else None,
        "2ps_subcode": second_hospital['Subcodice'] if second_hospital is not None else None,
        "2ps_latitude": second_hospital['latitude'] if second_hospital is not None else None,
        "2ps_longitude": second_hospital['longitude'] if second_hospital is not None else None,
        "travel_time_2ps": second_time
    }


async def process_chunk(session, chunk, hospitals, province_df, progress_bar, province_neighbors_dict):
    """Elabora un batch di centroidi in parallelo"""
    tasks = [process_centroid(session, row, hospitals, province_df, province_neighbors_dict) for _, row in chunk.iterrows()]
    results_batch = await asyncio.gather(*tasks)
    progress_bar.update(len(chunk))
    return [r for r in results_batch if r is not None]


def chunk_data(df, chunk_size):
    """Divide i dati in batch per ridurre overhead"""
    for i in range(0, len(df), chunk_size):
        yield df.iloc[i:i+chunk_size]

async def process_all_chunks(centroids_chunks, province_df, hospitals_df, province_neighbors_dict):
    """Elabora tutti i centroidi con sessione HTTP condivisa"""
    results = []
    total_chunks = len(centroids_chunks)

    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit_per_host=50)) as session:
        with tqdm(total=total_chunks, desc="Elaborazione Centroidi", unit="chunk") as progress_bar:
            for chunk in centroids_chunks:
                results_batch = await process_chunk(session, chunk, hospitals_df, province_df, progress_bar, province_neighbors_dict)
                results.extend(results_batch)

                # Salvataggio incrementale ogni 10000 centroidi
                if len(results) % 10000 == 0:
                    pd.DataFrame(results).to_csv("DATASET_DIST/risultati_parziali.csv", index=False)

    return results

def run_processing():
    """Funzione principale"""
    chunk_size = 5000  # Batch da 10000 per ridurre overhead
    centroids_chunks = list(chunk_data(centroids_df, chunk_size))

    # Preprocessa il dizionario delle province confinanti prima dell'elaborazione
    province_neighbors_dict = {}
    for _, row in province_df.iterrows():
        prov_sigla = row['Sigla_provincia']
        confinanti = row.iloc[1:].dropna().values.tolist()
        province_neighbors_dict[prov_sigla] = [prov_sigla] + confinanti

    # Passa il dizionario alle funzioni asincrone per l'elaborazione
    return asyncio.run(process_all_chunks(centroids_chunks, province_df, hospitals_df, province_neighbors_dict))


if __name__ == "__main__":
    results = run_processing()

    # Crea il DataFrame finale
    results_df = pd.DataFrame(results)

    # Salvataggio finale come CSV
    results_df.to_csv("DATASET_DIST/tempi_percorrenza_centroidi_ps.csv", index=False)

    # Creazione del GeoDataFrame per il salvataggio in formato GeoPackage
    gdf = gpd.GeoDataFrame(results_df, 
                           geometry=gpd.points_from_xy(results_df.grid_longitude, results_df.grid_latitude),
                           crs="EPSG:4326")  # Assicurati di usare il CRS corretto (WGS84 - EPSG:4326)

    # Salvataggio del GeoDataFrame in formato GeoPackage
    gdf.to_file("DATASET_DIST/tempi_percorrenza_centroidi_ps.gpkg", driver="GPKG")

    print("✅ Elaborazione completata con successo!")


**AEROPORTI**

PULIZIA DEI DATASET PER LA RICERCA

In [ ]:
# Evita errori di event loop in Jupyter
nest_asyncio.apply()

# Caricamento dati
centroids_df = pd.read_csv("DATASET_DIST/centroidi_parte_1.csv", dtype=str)
centroids_df.rename(columns=lambda x: x.strip(), inplace=True)
centroids_df.rename(columns={"provincia": "Sigla_provincia"}, inplace=True)

aeroporti_df = pd.read_excel("DATASET_DIST/Aeroporti_con_sigla_provincia.xlsx", dtype=str)
aeroporti_df.rename(columns=lambda x: x.strip(), inplace=True)
aeroporti_df.rename(columns={"SIGLA": "Sigla_provincia", "LAT": "latitude", "long": "longitude"}, inplace=True)
aeroporti_df["Sigla_provincia"] = aeroporti_df["Sigla_provincia"].str.strip()



# Conversione tipi
centroids_df['latitudine'] = centroids_df['latitudine'].astype(float)
centroids_df['longitudine'] = centroids_df['longitudine'].astype(float)
aeroporti_df['latitude'] = aeroporti_df['latitude'].astype(float)
aeroporti_df['longitude'] = aeroporti_df['longitude'].astype(float)

RICERCA DEGLI AEROPORTI PIU VICINI AL CENTROIDE

In [ ]:
# OSRM config
OSRM_URL = "http://localhost:5000/route/v1/driving"
SEMAPHORE = asyncio.Semaphore(50)
cache = {}
TIMEOUT = 600

async def get_travel_time(session, coords, max_retries=3):
    key = tuple(coords)
    if key in cache:
        return cache[key]

    url = f"{OSRM_URL}/{coords[1]},{coords[0]};{coords[3]},{coords[2]}?overview=false"

    for attempt in range(max_retries):
        async with SEMAPHORE:
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=TIMEOUT)) as response:
                    if response.status == 200:
                        data = orjson.loads(await response.read())
                        if 'routes' in data and data['routes']:
                            duration = data['routes'][0]['duration'] / 60
                            cache[key] = duration
                            return duration
            except (asyncio.TimeoutError, aiohttp.ClientError):
                pass

    return None

def haversine(lat1, lon1, lat2, lon2):
    # Calcolo della distanza in km tra due punti
    R = 6371  # Raggio della Terra in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

async def process_centroid(session, centroid, aeroporti):
    lat1, lon1, prov = centroid['latitudine'], centroid['longitudine'], centroid['Sigla_provincia']

    # Calcolo distanza geodetica a tutti gli aeroporti
    aeroporti['dist_km'] = aeroporti.apply(
        lambda row: haversine(lat1, lon1, row['latitude'], row['longitude']), axis=1
    )

    # Seleziona i 3 più vicini
    top8 = aeroporti.nsmallest(5, 'dist_km')

    coords_list = [(lat1, lon1, row['latitude'], row['longitude']) for _, row in top8.iterrows()]
    tasks = [get_travel_time(session, coords) for coords in coords_list]
    travel_times = await asyncio.gather(*tasks)

    min_time, min_idx = min(((t, i) for i, t in enumerate(travel_times) if t is not None), default=(None, None))

    if min_time is None:
        return {
            "grid_latitude": lat1,
            "grid_longitude": lon1,
            "grid_SIGLA_PROVINCIA": prov,
            "airport_name": None,
            "airport_latitude": None,
            "airport_longitude": None,
            "airport_prov": None,
            "travel_time_min": None
        }

    closest_airport = top8.iloc[min_idx]

    return {
        "grid_latitude": lat1,
        "grid_longitude": lon1,
        "grid_SIGLA_PROVINCIA": prov,
        "airport_name": closest_airport['NOME'],
        "airport_latitude": closest_airport['latitude'],
        "airport_longitude": closest_airport['longitude'],
        "airport_prov": closest_airport['Sigla_provincia'],
        "travel_time_min": min_time
    }


async def process_all_centroids(centroids_df, aeroporti_df):
    results = []
    chunk_size = 20000
    chunks = [centroids_df.iloc[i:i + chunk_size] for i in range(0, len(centroids_df), chunk_size)]

    first_chunk = True
    async with aiohttp.ClientSession() as session:
        for chunk_idx, chunk in enumerate(tqdm(chunks, desc="🔄 Elaborazione chunks da 10.000")):
            tasks = [process_centroid(session, row, aeroporti_df) for _, row in chunk.iterrows()]
            results_batch = await asyncio.gather(*tasks)
            results.extend(results_batch)

            # Save partial CSV (append mode)
            df_batch = pd.DataFrame(results_batch)
            write_mode = 'w' if first_chunk else 'a'
            header_flag = first_chunk
            df_batch.to_csv("DATASET_DIST/partial_dataset_aeroporti.csv", mode=write_mode, index=False, header=header_flag)
            first_chunk = False

    return results

def run_processing():
    return asyncio.run(process_all_centroids(centroids_df, aeroporti_df))

if __name__ == "__main__" or True:  # Permette l'esecuzione anche in notebook
    results = run_processing()

    # DataFrame finale
    results_df = pd.DataFrame(results)

    # GeoDataFrame e salvataggio
    gdf = gpd.GeoDataFrame(results_df,
                           geometry=gpd.points_from_xy(results_df.grid_longitude, results_df.grid_latitude),
                           crs="EPSG:4326")

    gdf.to_file("DATASET_DIST/tempi_percorrenza_centroidi_aeroporti.gpkg", driver="GPKG")
    results_df.to_csv("DATASET_DIST/tempi_percorrenza_centroidi_aeroporti.csv", index=False)
    print("✅ Elaborazione completata con successo!")


## **PORTI**

PULIZIA DEI DASASET PER LA RICERCA

In [ ]:
# Evita errori di event loop in Jupyter
nest_asyncio.apply()

# Caricamento dati
centroids_df = pd.read_csv("DATASET_DIST/centroidi_con_provincia.csv", dtype=str)
centroids_df.rename(columns=lambda x: x.strip(), inplace=True)
centroids_df.rename(columns={"provincia": "Sigla_provincia"}, inplace=True)

porti_df = pd.read_csv("DATASET_DIST/porti_provincia.csv", dtype=str)
porti_df.rename(columns=lambda x: x.strip(), inplace=True)
porti_df.rename(columns={"SIGLA_PROVINCIA": "Sigla_provincia", "lat": "latitude", "lon": "longitude"}, inplace=True)
porti_df["Sigla_provincia"] = porti_df["Sigla_provincia"].str.strip()



# Conversione tipi
centroids_df['latitudine'] = centroids_df['latitudine'].astype(float)
centroids_df['longitudine'] = centroids_df['longitudine'].astype(float)
porti_df['latitude'] = porti_df['latitude'].astype(float)
porti_df['longitude'] = porti_df['longitude'].astype(float)

RICERCA DEI PORTI PIU VICINI AL CENTROIDE

In [ ]:
# OSRM config
OSRM_URL = "http://localhost:5000/route/v1/driving"
SEMAPHORE = asyncio.Semaphore(100)
cache = {}
TIMEOUT = 600

async def get_travel_time(session, coords, max_retries=3):
    key = tuple(coords)
    if key in cache:
        return cache[key]

    url = f"{OSRM_URL}/{coords[1]},{coords[0]};{coords[3]},{coords[2]}?overview=false"

    for attempt in range(max_retries):
        async with SEMAPHORE:
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=TIMEOUT)) as response:
                    if response.status == 200:
                        data = orjson.loads(await response.read())
                        if 'routes' in data and data['routes']:
                            duration = data['routes'][0]['duration'] / 60
                            cache[key] = duration
                            return duration
            except (asyncio.TimeoutError, aiohttp.ClientError):
                pass

    return None

def haversine(lat1, lon1, lat2, lon2):
    # Calcolo della distanza in km tra due punti
    R = 6371  # Raggio della Terra in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

async def process_centroid(session, centroid, aeroporti):
    lat1, lon1, prov = centroid['latitudine'], centroid['longitudine'], centroid['Sigla_provincia']

    # Calcolo distanza geodetica a tutti gli aeroporti
    aeroporti['dist_km'] = aeroporti.apply(
        lambda row: haversine(lat1, lon1, row['latitude'], row['longitude']), axis=1
    )

    # Seleziona i 3 più vicini
    top8 = aeroporti.nsmallest(5, 'dist_km')

    coords_list = [(lat1, lon1, row['latitude'], row['longitude']) for _, row in top8.iterrows()]
    tasks = [get_travel_time(session, coords) for coords in coords_list]
    travel_times = await asyncio.gather(*tasks)

    min_time, min_idx = min(((t, i) for i, t in enumerate(travel_times) if t is not None), default=(None, None))

    if min_time is None:
        return {
            "grid_latitude": lat1,
            "grid_longitude": lon1,
            "grid_SIGLA_PROVINCIA": prov,
            "port_name": None,
            "port_latitude": None,
            "port_longitude": None,
            "port_prov": None,
            "travel_time_min": None
        }

    closest_airport = top8.iloc[min_idx]

    return {
        "grid_latitude": lat1,
        "grid_longitude": lon1,
        "grid_SIGLA_PROVINCIA": prov,
        "port_name": closest_airport['nome'],
        "port_latitude": closest_airport['latitude'],
        "port_longitude": closest_airport['longitude'],
        "port_prov": closest_airport['Sigla_provincia'],
        "travel_time_min": min_time
    }


async def process_all_centroids(centroids_df, porti_df):
    results = []
    chunk_size = 20000
    chunks = [centroids_df.iloc[i:i + chunk_size] for i in range(0, len(centroids_df), chunk_size)]

    first_chunk = True
    async with aiohttp.ClientSession() as session:
        for chunk_idx, chunk in enumerate(tqdm(chunks, desc="🔄 Elaborazione chunks da 10.000")):
            tasks = [process_centroid(session, row, porti_df) for _, row in chunk.iterrows()]
            results_batch = await asyncio.gather(*tasks)
            results.extend(results_batch)

            # Save partial CSV (append mode)
            df_batch = pd.DataFrame(results_batch)
            write_mode = 'w' if first_chunk else 'a'
            header_flag = first_chunk
            df_batch.to_csv("DATASET_DIST/partial_dataset_aeroporti.csv", mode=write_mode, index=False, header=header_flag)
            first_chunk = False

    return results

def run_processing():
    return asyncio.run(process_all_centroids(centroids_df, porti_df))

if __name__ == "__main__" or True:  # Permette l'esecuzione anche in notebook
    results = run_processing()

    # DataFrame finale
    results_df = pd.DataFrame(results)

    # GeoDataFrame e salvataggio
    gdf = gpd.GeoDataFrame(results_df,
                           geometry=gpd.points_from_xy(results_df.grid_longitude, results_df.grid_latitude),
                           crs="EPSG:4326")

    gdf.to_file("DATASET_DIST/tempi_percorrenza_centroidi_porti.gpkg", driver="GPKG")
    results_df.to_csv("DATASET_DIST/tempi_percorrenza_centroidi_porti.csv", index=False)
    print("✅ Elaborazione completata con successo!")


In [ ]:
import pandas as pd
import geopandas as gpd

# Percorsi dei CSV da unire
file1 = "DATASET_DIST/tempi_percorrenza_centroidi_ps.csv"
file2 = "DATASET_DIST/tempi_percorrenza_centroidi_2.csv"

# Leggi i CSV
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Unione dei CSV
df_unito = pd.concat([df1, df2], ignore_index=True)
df_unito.to_csv("DATASET_DIST/tempi_percorrenza_ps.csv", index=False)
print("✅ CSV uniti in 'centroidi_unificati.csv'")

# Percorsi dei GeoPackage da unire
gpk1 = "DATASET_DIST/tempi_percorrenza_centroidi_ps.gpkg"
gpk2 = "DATASET_DIST/tempi_percorrenza_centroidi_ps_2.gpkg"

# Leggi i GPKG
gdf1 = gpd.read_file(gpk1)
gdf2 = gpd.read_file(gpk2)

# Unione dei GeoDataFrame
gdf_unito = pd.concat([gdf1, gdf2], ignore_index=True)

# Salva in nuovo file GPKG
gdf_unito.to_file("DATASET_DIST/tempi_percorrenza_ps.gpkg", driver="GPKG")
print("✅ GPKG uniti in 'tempi_unificati.gpkg'")
